# Qwen3 VL AutoTagger CLI - Colab T4 Diagnostic

This notebook collects full diagnostics into one text file, even when errors happen.

What you do:
1. Run all cells top-to-bottom.
2. Upload 1-3 test images when prompted.
3. Download `colab_t4_diagnostic_report.txt` from the last cell and send it here.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import os
import platform
import subprocess
import sys
import traceback

from google.colab import files

NL = chr(10)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
REPORT_PATH = Path('/content/colab_t4_diagnostic_report.txt')
REPO_URL = 'https://github.com/ekkonwork/qwen3-vl-autotagger-cli.git'
WORKDIR = Path('/content/qwen3-vl-autotagger-cli')
INPUT_DIR = Path('/content/input_images')
OUTPUT_DIR = Path('/content/outputs')

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text('', encoding='utf-8')

def append_section(title: str, content: str) -> None:
    stamp = datetime.now(timezone.utc).isoformat()
    block = f"{NL}===== {title} @ {stamp} ====={NL}{content}{NL}"
    with REPORT_PATH.open('a', encoding='utf-8') as f:
        f.write(block)

def run_cmd(cmd, cwd=None, timeout=1800):
    if isinstance(cmd, str):
        shell = True
        cmd_display = cmd
    else:
        shell = False
        cmd_display = ' '.join(cmd)

    try:
        res = subprocess.run(
            cmd,
            cwd=cwd,
            shell=shell,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        content = (
            f"CMD: {cmd_display}{NL}"
            f"CWD: {cwd or os.getcwd()}{NL}"
            f"EXIT: {res.returncode}{NL}"
            f"--- STDOUT ---{NL}{res.stdout}{NL}"
            f"--- STDERR ---{NL}{res.stderr}{NL}"
        )
        append_section('COMMAND', content)
        return res
    except Exception:
        append_section('COMMAND_EXCEPTION', f"CMD: {cmd_display}{NL}{traceback.format_exc()}")
        class Dummy:
            returncode = 999
            stdout = ''
            stderr = traceback.format_exc()
        return Dummy()

append_section('SESSION_START', f'run_id={RUN_ID}')
print('Report path:', REPORT_PATH)


In [ ]:
try:
    base_info = [
        f"python={sys.version}",
        f"platform={platform.platform()}",
        f"machine={platform.machine()}",
        f"processor={platform.processor()}",
        f"cwd={os.getcwd()}",
        f"PATH={os.environ.get('PATH', '')}",
    ]
    append_section('BASE_INFO', NL.join(base_info))

    run_cmd(['python', '--version'])
    run_cmd(['pip', '--version'])
    run_cmd('nvidia-smi')
    run_cmd('free -h')
    run_cmd('df -h')

    try:
        import torch
        torch_info = [
            f"torch.__version__={torch.__version__}",
            f"cuda_available={torch.cuda.is_available()}",
            f"cuda_device_count={torch.cuda.device_count()}",
        ]
        if torch.cuda.is_available():
            torch_info.extend([
                f"cuda_device_name={torch.cuda.get_device_name(0)}",
                f"cuda_capability={torch.cuda.get_device_capability(0)}",
            ])
        append_section('TORCH_INFO', NL.join(torch_info))
    except Exception:
        append_section('TORCH_INFO_EXCEPTION', traceback.format_exc())
except Exception:
    append_section('BASE_COLLECTION_EXCEPTION', traceback.format_exc())

print('Base diagnostics collected.')


In [ ]:
try:
    if WORKDIR.exists():
        append_section('REPO', f'Repo already exists: {WORKDIR}')
        run_cmd(['git', '-C', str(WORKDIR), 'status'])
        run_cmd(['git', '-C', str(WORKDIR), 'pull', '--ff-only'])
    else:
        run_cmd(['git', 'clone', REPO_URL, str(WORKDIR)], timeout=1200)

    run_cmd([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'], timeout=1800)
    run_cmd([sys.executable, '-m', 'pip', 'install', '-r', str(WORKDIR / 'requirements.txt')], timeout=5400)
    run_cmd([sys.executable, str(WORKDIR / 'install.py')], cwd=str(WORKDIR), timeout=1200)
    run_cmd([sys.executable, '-m', 'pip', 'freeze'], timeout=1800)
except Exception:
    append_section('INSTALL_EXCEPTION', traceback.format_exc())

print('Repository/dependency step completed (check report for details).')


In [ ]:
try:
    print('Upload test images (1-3 files is enough):')
    uploaded = files.upload()

    saved = []
    for name, data in uploaded.items():
        out_path = INPUT_DIR / name
        with open(out_path, 'wb') as f:
            f.write(data)
        saved.append(str(out_path))

    append_section('UPLOADED_FILES', NL.join(saved) if saved else 'No files uploaded')
    print('Saved files:', len(saved))
except Exception:
    append_section('UPLOAD_EXCEPTION', traceback.format_exc())
    print('Upload failed, see report.')


In [ ]:
CLI_EXIT_CODE = None
try:
    image_files = [p for p in INPUT_DIR.iterdir() if p.is_file()]
    if not image_files:
        append_section('CLI_SKIP', f'No images found in {INPUT_DIR}')
        print('No images uploaded. CLI step skipped.')
    else:
        cli_cmd = [
            sys.executable, '-m', 'qwen3_vl_autotagger_cli.cli',
            str(INPUT_DIR),
            '--recursive',
            '--output-dir', str(OUTPUT_DIR),
            '--output-format', 'jpg',
            '--file-prefix', 'diag',
            '--metadata-jsonl', str(OUTPUT_DIR / 'metadata.jsonl'),
            '--write-xmp',
            '--require-exiftool',
            '--log-tags',
            '--attempts', '2',
            '--max-keywords', '50',
            '--max-new-tokens', '600',
            '--temperature', '0.7',
            '--top-p', '0.9',
            '--repetition-penalty', '1.15',
            '--min-pixels', str(256 * 28 * 28),
            '--max-pixels', str(756 * 756),
            '--allow-resize',
        ]
        result = run_cmd(cli_cmd, cwd=str(WORKDIR), timeout=10800)
        CLI_EXIT_CODE = result.returncode
        append_section('CLI_EXIT', f'CLI return code: {CLI_EXIT_CODE}')
        print('CLI finished with code:', CLI_EXIT_CODE)
except Exception:
    append_section('CLI_EXCEPTION', traceback.format_exc())
    print('CLI execution failed, see report.')


In [ ]:
try:
    output_files = sorted(str(p) for p in OUTPUT_DIR.glob('*') if p.is_file())
    append_section('OUTPUT_FILES', NL.join(output_files) if output_files else 'No output files')

    metadata_path = OUTPUT_DIR / 'metadata.jsonl'
    if metadata_path.exists():
        lines_ = metadata_path.read_text(encoding='utf-8', errors='ignore').splitlines()
        preview = NL.join(lines_[:10])
        append_section('METADATA_PREVIEW', preview if preview else 'metadata.jsonl exists but empty')
    else:
        append_section('METADATA_PREVIEW', 'metadata.jsonl not found')

    run_cmd(['bash', '-lc', f'ls -lah {OUTPUT_DIR}'])
except Exception:
    append_section('POSTCHECK_EXCEPTION', traceback.format_exc())

print('Post-run checks completed.')


In [ ]:
try:
    final = [
        f'run_id={RUN_ID}',
        f'cli_exit_code={CLI_EXIT_CODE}',
        f'workdir={WORKDIR}',
        f'input_dir={INPUT_DIR}',
        f'output_dir={OUTPUT_DIR}',
    ]
    append_section('FINAL_SUMMARY', NL.join(final))
except Exception:
    append_section('FINAL_SUMMARY_EXCEPTION', traceback.format_exc())

print('Diagnostic report ready:', REPORT_PATH)
files.download(str(REPORT_PATH))
